# Hyperparameter Tuning


Vocabulary: the parameters of machine learning models that can be modified are called hyperparameters. Be careful not to confuse them with model parameters, which are calculated automatically during training.

Example: the number of layers in a neural network is a hyperparameter; the bias of a given neuron is a network parameter.

Hyperparameter tuning methods: Grid Search, Random Search, etc.

https://larevueia.fr/3-methodes-pour-optimiser-les-hyperparametres-de-vos-modeles-de-machine-learning/

In this study, we will look at a tuning example for the SVC method. The database: Titanic!


0. Import libraries:
- pandas: used for data manipulation and analysis.
- train_test_split: Sklearn library for splitting arrays or matrices into random training and test subsets.
- GridSearchCV: Sklearn library for exhaustive search over specified parameter values for an estimator.
- RandomizedSearchCV: Sklearn library for random search over hyperparameters.
- svm: Sklearn Support Vector Machines library.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay, classification_report

plots_dir = Path("../../plots/etude6_hyperparametres")
plots_dir.mkdir(parents=True, exist_ok=True)


1. Load the database "titanic.csv".


In [ ]:
df = pd.read_csv("titanic.csv")
df.head()


2. a) Preprocess the database, for example by removing data without a "Survived" label or useless variables.

In this study, we will keep only quantitative variables.


In [ ]:
df_clean = df.dropna(subset=["Survived"]).copy()
quantitative_columns = ["Pclass", "Age", "SibSp", "Parch", "Fare"]
model_df = df_clean[["Survived"] + quantitative_columns].dropna()
X = model_df[quantitative_columns]
y = model_df["Survived"]
print(model_df.shape)
model_df.head()


2. b) Analyze the database.

You are free to give the answer. Suggestions: for example, display data statistics, a histogram, a pairplot, ...


In [ ]:
display(model_df.info())
display(model_df.describe())
print("Missing values:")
display(model_df.isna().sum())

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
axes = axes.ravel()
for ax, col in zip(axes, model_df.columns):
    sns.histplot(model_df[col], kde=True, ax=ax)
    ax.set_title(col)
for ax in axes[len(model_df.columns):]:
    ax.set_visible(False)
fig.suptitle("Titanic Quantitative Variable Distributions", y=1.02)
fig.tight_layout()
fig.savefig(plots_dir / "01_quantitative_distributions.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(model_df.corr(numeric_only=True), annot=True, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Titanic Correlation Heatmap")
fig.tight_layout()
fig.savefig(plots_dir / "02_correlation_heatmap.pdf", bbox_inches="tight")
plt.show()


## Without Hyperparameter Tuning


3. Perform train_test_split.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
print(X_train.shape, X_test.shape)


4. Fit the SVC model and display the score on "test".


In [ ]:
baseline_svc = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(random_state=0)),
])
baseline_svc.fit(X_train, y_train)
y_pred_baseline = baseline_svc.predict(X_test)
print(f"Baseline SVC test accuracy: {accuracy_score(y_test, y_pred_baseline):.4f}")
print(classification_report(y_test, y_pred_baseline))

cm = confusion_matrix(y_test, y_pred_baseline)
disp = ConfusionMatrixDisplay(cm, display_labels=["Did not survive", "Survived"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("Baseline SVC Confusion Matrix")
fig.tight_layout()
fig.savefig(plots_dir / "03_baseline_svc_confusion_matrix.pdf", bbox_inches="tight")
plt.show()


## Hyperparameter Tuning

5. What are the hyperparameters of an SVC model?


Answer: Important SVC hyperparameters include C, kernel, gamma, degree for polynomial kernels, coef0 for polynomial/sigmoid kernels, class_weight, shrinking, probability, and tolerance. C controls regularization, kernel defines the decision function shape, and gamma controls the influence of individual samples for rbf/poly/sigmoid kernels.


6. Create a grid for these hyperparameters and run GridSearchCV.


In [ ]:
parameters = {
    "svc__C": [0.01, 0.1, 1, 5, 10],
    "svc__kernel": ["linear", "rbf"],
    "svc__gamma": ["scale", "auto"],
}

grid_search = GridSearchCV(
    estimator=Pipeline([("scaler", StandardScaler()), ("svc", SVC(random_state=0))]),
    param_grid=parameters,
    cv=5,
    scoring="accuracy",
    n_jobs=-1,
)
grid_search.fit(X_train, y_train)
grid_search


7. Display the best hyperparameters and the corresponding score.


In [ ]:
print("Best GridSearchCV parameters:")
print(grid_search.best_params_)
print(f"Best cross-validation score: {grid_search.best_score_:.4f}")
print(f"Test score with best estimator: {grid_search.score(X_test, y_test):.4f}")


8. Redo the searches with RandomizedSearchCV.


In [ ]:
random_parameters = {
    "svc__C": np.logspace(-2, 2, 20),
    "svc__kernel": ["linear", "rbf"],
    "svc__gamma": ["scale", "auto"],
}

random_search = RandomizedSearchCV(
    estimator=Pipeline([("scaler", StandardScaler()), ("svc", SVC(random_state=0))]),
    param_distributions=random_parameters,
    n_iter=12,
    cv=5,
    scoring="accuracy",
    random_state=0,
    n_jobs=-1,
)
random_search.fit(X_train, y_train)
random_search


9. Display the best hyperparameters and the corresponding score.


In [ ]:
print("Best RandomizedSearchCV parameters:")
print(random_search.best_params_)
print(f"Best cross-validation score: {random_search.best_score_:.4f}")
print(f"Test score with best estimator: {random_search.score(X_test, y_test):.4f}")

results = pd.DataFrame({
    "model": ["Baseline SVC", "GridSearchCV SVC", "RandomizedSearchCV SVC"],
    "test_accuracy": [baseline_svc.score(X_test, y_test), grid_search.score(X_test, y_test), random_search.score(X_test, y_test)],
})
display(results)

fig, ax = plt.subplots(figsize=(7, 5))
sns.barplot(data=results, x="model", y="test_accuracy", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("SVC Test Accuracy Comparison")
ax.tick_params(axis="x", rotation=20)
fig.tight_layout()
fig.savefig(plots_dir / "04_svc_accuracy_comparison.pdf", bbox_inches="tight")
plt.show()


10. Your conclusion:


Answer: Hyperparameter tuning improves or validates the SVC configuration by testing several combinations with cross-validation. GridSearchCV explores the full specified grid, while RandomizedSearchCV samples combinations and is often faster when the search space is larger.
